In [5]:
import torch as T
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import sys
from gym import Env
from gym.spaces import Discrete, Box

import numpy as np
import rlmod as rk
import scipy.linalg as la
import csv
import cmath as cm
import numpy as np
from rl_torch import MyEnv
from scipy.linalg import expm,norm
import itertools

In [4]:
# hamiltonianos usando el codigo del paper
nc = 3
STATE_NUM = 6 #the system has 6 spins

MAG = 100  #other parameters
COUPLING = 1
DT = 0.15

####################################### defining action, each row of 'mag' corresponds to one allowed configuration of magnetic
def binact(A):                #action label

    m = np.zeros(nc)
    for ii in range(nc):  # transfer action to a binary list, for example: action=5, x=[1, 0, 1, 0], the first and third magnetic is on
        m[nc - 1 - ii] = A >= 2 ** (nc - 1 - ii)
        A = A - 2 ** (nc - 1 - ii) * m[nc - 1 - ii]
    return(m)

mag=[]
for ii in range(8):  #control at the beginning
    mag.append( list( np.concatenate((binact(ii)* MAG,np.zeros(STATE_NUM -nc))) ))

for ii in range(1,8): #control at the end
    mag.append( list( np.concatenate((np.zeros(STATE_NUM -nc),binact(ii)* MAG)) ))

mag.append([MAG for ii in range(STATE_NUM)])
########################################

# acciones como las defini yo
acc_zhang = np.zeros((16, STATE_NUM, STATE_NUM),dtype=np.complex_)
acc_sofi = rk.acciones(MAG,STATE_NUM)

for i in np.arange(0,16):
    acc_zhang[i,:,:] = np.diag([COUPLING for i in range(STATE_NUM-1)], 1)*(1-0j) + np.diag([COUPLING for i in range(STATE_NUM-1)], -1)*(1+0j) + np.diag(mag[i])
    diff = acc_sofi[i,:,:] - acc_zhang[i,:,:]
    nz = np.count_nonzero(diff)
    print(nz, ' different elements')


0  different elements
0  different elements
0  different elements
0  different elements
0  different elements
0  different elements
0  different elements
0  different elements
0  different elements
0  different elements
0  different elements
0  different elements
0  different elements
0  different elements
0  different elements
0  different elements


In [6]:
env = MyEnv(STATE_NUM)
props = env.propagadores

Descomposicion espectral: correcta
Propagacion de autoestados: correcta


/home/sofi/.local/lib/python3.11/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


In [44]:
# compare evolution for all actions

psi = [0 for i in range(STATE_NUM)]  #initial state is [1;0;0;0;0...]
psi[0] = 1
state = np.array([str(i) for i in psi])
state = np.array(list(itertools.chain(*[(i.real, i.imag) for i in psi])))
cstate = [complex(state[2*i], state[2*i+1]) for i in range(STATE_NUM)]  #transfer real vector to complex vector

sstate = np.zeros(2*STATE_NUM)
csstate = np.zeros(STATE_NUM, dtype=np.complex_)

sstate[0] = 1

j = 0

for i in np.arange(0,STATE_NUM):
            csstate[i] = complex(sstate[j],sstate[j+1])
            j+=2


for i in np.arange(0,16):
        statelist = np.transpose(np.mat(cstate))   #to 'matrix'
        next_zstate = expm(-1j*acc_zhang[i,:,:]*DT)*statelist 
        next_sstate = np.matmul(props[i, :, :], csstate)

print(next_zstate)
print(next_sstate)
print(np.real(next_sstate[STATE_NUM-1]*np.conjugate(next_sstate[STATE_NUM-1])))
print((abs(next_zstate[-1])**2)[0,0])



[[-7.51173413e-01-6.42999485e-01j]
 [-9.68136567e-02+1.13100938e-01j]
 [ 8.49852305e-03+7.27467965e-03j]
 [ 3.64143952e-04-4.25405093e-04j]
 [-1.59646661e-05-1.36656488e-05j]
 [-4.10409322e-07+4.79453839e-07j]]
[-7.51173413e-01-6.42999485e-01j -9.68136567e-02+1.13100938e-01j
  8.49852305e-03+7.27467965e-03j  3.64143952e-04-4.25405093e-04j
 -1.59646661e-05-1.36656488e-05j -4.10409320e-07+4.79453842e-07j]
3.983117970494534e-13
3.983117954360455e-13


In [46]:
test_sequence = [8, 3, 4, 14, 11, 7, 8, 0, 15, 7, 12, 7, 8, 15, 9, 2, 2, 3, 3, 3, 3, 1, 2, 2, 1, 5, 15, 3, 3, 2, 10, 11, 6, 15, 1, 13, 10, 10, 6, 13, 15, 9,
                   8, 15, 2, 0, 4, 2, 6, 0, 3, 4, 0, 7, 4, 4, 10, 13, 10, 13, 10, 10, 3, 4, 2, 5, 4, 10, 10, 4, 3, 1, 3, 10, 10, 9, 4, 2, 2, 2, 12, 3, 11, 2, 10, 4, 7, 8, 5, 11]

# init zhang state
psi = [0 for i in range(STATE_NUM)]  #initial state is [1;0;0;0;0...]
psi[0] = 1
state = np.array([str(i) for i in psi])
state = np.array(list(itertools.chain(*[(i.real, i.imag) for i in psi])))
cstate = [complex(state[2*i], state[2*i+1]) for i in range(STATE_NUM)]  #transfer real vector to complex vector

next_zstate = np.transpose(np.mat(cstate))   #to 'matrix'

# init mi estado
sstate = np.zeros(2*STATE_NUM)
csstate = np.zeros(STATE_NUM, dtype=np.complex_)
sstate[0] = 1

j = 0
for i in np.arange(0,STATE_NUM):
            csstate[i] = complex(sstate[j],sstate[j+1])
            j+=2

next_sstate = csstate

for action in test_sequence:
        next_zstate = expm(-1j*acc_zhang[action,:,:]*DT)*next_zstate 
        next_sstate = np.matmul(props[action, :, :], next_sstate)

print(np.real(next_sstate[STATE_NUM-1]*np.conjugate(next_sstate[STATE_NUM-1])))
print((abs(next_zstate[-1])**2)[0,0])        

0.47346621361211105
0.47346621361229657
